# Setup
[Time-Weighted Asset Price](https://en.wikipedia.org/wiki/Time-weighted_average_price) of a security (tradable asset) over time. 

TWAP orders are a strategy of executing trades evenly over a specified time period.

$$
P_{\mathrm{TWAP}} = \frac{\sum_{j} P_{j} \cdot T_{j}}{\sum_{j} T_{j}}
$$

where:

- $ P_{\mathrm{TWAP}} $ — Time Weighted Average Price
- $ P_{j} $ — Price of the security at measurement time $ j $  
- $ T_{j} $ — Time interval since the previous measurement $ j $  
- $ j $ — Each individual measurement that occurs over the defined time period  

> Increasing the total measurement period $ \sum_{j} T_{j} $ results in a less up-to-date (more lagging) price.


In [2]:
#load basic libraries
import polars as pl  # It is advised to use polars, as the detasets can be quite memory-intensive
from datetime import timedelta
from utils import *
from data.simulate_walk_the_book import *
import re, warnings

%load_ext autoreload
%autoreload 2
%matplotlib inline

tol = 1e-6 # displayed numbers are rounded

In [3]:
# Query data
# We load the BTCUSDT pair
data_direct = "data/BTCUSDT"
X_train = pl.read_parquet(f"{data_direct}/X_train.parquet")
y_train = pl.read_parquet(f"{data_direct}/y_train.parquet")
X_test = pl.read_parquet(f"{data_direct}/X_test.parquet")

# load the volume to fill info from the text file
with open(f"{data_direct}/vol_to_fill.txt", "r") as file:
    content = file.read().strip()

match = re.search(r"The volume to fill is: ([\d.]+)", content)
if match:
    volume_to_fill  = float(match.group(1))
    print(f"Extracted number: {volume_to_fill}")
else:
    print("No number found in the file.")

Extracted number: 4.0


# TWAP

In [4]:
minute = y_train['time_in_hour'].unique().sort()
ids = y_train['anonymized_id'].unique().sort()

submission = ids.to_frame().join(
        minute.to_frame(),
        how="cross"
    ).with_columns(
    position=pl.lit(volume_to_fill/60).alias("position")
)

submission

anonymized_id,time_in_hour,position
u64,duration[ms],f64
10076153343292355,59m,0.066667
10076153343292355,59m 1s,0.066667
10076153343292355,59m 2s,0.066667
10076153343292355,59m 3s,0.066667
10076153343292355,59m 4s,0.066667
…,…,…
18444992991527050644,59m 55s,0.066667
18444992991527050644,59m 56s,0.066667
18444992991527050644,59m 57s,0.066667


# Walk the Book

In [5]:
id = 

positions_np = submission["position"].to_numpy()

ask_prices_np = y_train.select(
    [f"ask_price_{i}" for i in range(1, 6)]
).to_numpy()

ask_vols_np = y_train.select(
    [f"ask_vol_{i}" for i in range(1, 6)]
).to_numpy()

bid_prices_np = y_train.select(
    [f"bid_price_{i}" for i in range(1, 6)]
).to_numpy()

bid_vols_np = y_train.select(
    [f"bid_vol_{i}" for i in range(1, 6)]
).to_numpy()

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    walk = simulate_walk_the_book(positions_np, ask_prices_np, ask_vols_np, bid_prices_np, bid_vols_np)

SyntaxError: invalid syntax (404567520.py, line 1)

# Score Function

In [ ]:
def objective_func(avg_price, close_price):
    return abs(avg_price - close_price) / close_price

def volume_penalty(volume_to_fill, total_vol_exec):
    return min(100, volume_to_fill/total_vol_exec)

def score(avg_price, total_vol_exec, close_price, volume_to_fill):
    return objective_func(avg_price, close_price) * volume_penalty(volume_to_fill, total_vol_exec)

def score(walk, close_price, volume_to_fill):
    return score(walk[0], walk[1], close_price, volume_to_fill)

In [ ]:
score(walk, y_train.select())

In [ ]:
submission

NameError: name 'submission' is not defined

In [ ]:
walk

(np.float64(2819.9999999989072), np.float64(102736.470780889))

In [ ]:
submission

anonymized_id,time_in_hour,position
u64,duration[ms],f64
10076153343292355,59m,0.066667
10076153343292355,59m 1s,0.066667
10076153343292355,59m 2s,0.066667
10076153343292355,59m 3s,0.066667
10076153343292355,59m 4s,0.066667
…,…,…
18444992991527050644,59m 55s,0.066667
18444992991527050644,59m 56s,0.066667
18444992991527050644,59m 57s,0.066667


you calculate the score for a particular hour. average scores for all hours.